**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/hiraoka-2025"], check=True)
    os.chdir("/content/encore/entries/hiraoka-2025")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Hiraoka & Jo (2025): the dynamic generative algorithms (Section IV, Figure 4)

Paper: T. Hiraoka and H.-H. Jo, "Hierarchical organization of bursty trains in event
sequences", doi:10.1063/5.0296506, read from the arXiv preprint body under `paper/`.
These claims are `data: elsewhere` in the claims file: their data are not deposited and are
generated here by the paper's Algorithms 1 and 2, for which no code is linked. They do not use
DANDI:000044 (version 0.210812.1516, the subject of `reproduce_hiraoka_2025.ipynb`); the helper
names it, and the sanity cell below only checks that the pinned version is reachable.

Set-up from Section IV, paragraph 9: n = 10^6 events, gamma = 2.5, delta_0 = 1, uniform IETs
(Eq. 10), linear (Eq. 12) and quadratic (Eq. 13) timescales with w = 100, exponential (Eq. 14)
with w = 2, merging numbers zeta-distributed (Eq. 15) with the memory function of Eq. 16 for the
incremental algorithm (Algorithm 2), which Figure 4 uses.

Claims tested here: fig4-model-heavy-tails, fig4-model-iet-shapes, model-decremental-equivalent.

In [ ]:
import time, resource, zlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp
import powerlaw
import bursttree as bt

T0 = time.time()
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_rows", 300)
PAPER = "hiraoka-2025"
N, GAMMA = 10**6, 2.5
SERIES = ("linear", "quadratic", "exponential")
# Figure 4 legends: merging-number pairs delta = r * delta' per series.
RATIOS = {"linear": (1.8, 2.4, 3.3), "quadratic": (3.3, 5.9, 11.0), "exponential": (5.3, 12.0, 28.0)}
OFFSETS = (0, 1000, 2000, 3000, 4000)
def seed_of(claim):
    return zlib.crc32(f"{PAPER}/{claim}".encode())
print("powerlaw", powerlaw.__version__)

In [ ]:
print("dandiset", bt.DANDISET, "version", bt.DANDI_VERSION, "(not used by these claims)")
print("assets at the pinned version:", len(bt.list_assets()))
for s in SERIES:
    print(s, "delta_l for l = 0..5:", bt.timescales(s, 5))
print("memory function Eq. 16, q(1..4):", np.round([(k / (k + 1)) ** (GAMMA - 1) for k in range(1, 5)], 4))
print("Eq. 9 applied to the zeta distribution, q(1..4):", np.round(bt._q_exact_zeta(GAMMA)[1:5], 4))
# what each choice implies for P(k): mean merging number, from 10^6 samples
rng = np.random.default_rng(seed_of("model-decremental-equivalent") + 99)
for dist in ("zeta", "eq16"):
    k = bt.sample_merging(10**6, GAMMA, rng, dist)
    print(f"P(k) {dist}: P(k=1) = {(k == 1).mean():.4f}, mean k = {k.mean():.3f}")
t1 = time.time(); _ = bt.simulate_incremental(10**5, "linear", GAMMA, np.random.default_rng(0)); print(f"time for 1e5 events: {time.time() - t1:.2f} s")

## fig4-model-heavy-tails

> Event sequences generated by the incremental dynamic algorithm, with linear, quadratic or
> exponential timescale series, show heavy-tailed burst-size and merging-number distributions
> over a wide range of timescales, with stable exponents.

Criterion: Algorithm 2 with the set-up above; bursts, fit and test as for Figure 2 (O3, O4); the
pairs of the Figure 4 legends. Reproduced when, for each series, the power law is supported at
more than half of the relevant timescales and pairs, and the beta and gamma estimates each have
an interquartile range below 0.5. Under the claim's seed both grids and all four fit readings are
run, with the exponent unbounded, and on grid (a) also with the exponent capped at 3 (powerlaw
2.0.0 default, a variant); under the four further seeds, grid (a) with the four unbounded readings
(a choice for run time).

In [ ]:
def judge_series(res):
    B = pd.DataFrame(res["bursts"]); B = B[B.on_grid]; P = pd.DataFrame(res["pairs"])
    out = []
    for r, b in B.groupby("reading"):
        p = P[P.reading == r]
        iqr = lambda x: float(np.subtract(*np.percentile(x, [75, 25]))) if len(x) else np.nan
        out.append(dict(reading=r, n_at_cap=int(b.at_cap.sum() + p.at_cap.sum()), n_delta=len(b), b_supported=int(b.supported.sum()), n_pairs=len(p), k_supported=int(p.supported.sum()),
                        beta_median=round(b.alpha.median(), 3), beta_iqr=round(iqr(b.alpha), 3),
                        gamma_median=round(p.alpha.median(), 3) if len(p) else np.nan, gamma_iqr=round(iqr(p.alpha), 3),
                        holds=bool(b.supported.sum() > len(b) / 2 and len(p) > 0 and p.supported.sum() > len(p) / 2
                                   and iqr(b.alpha) < 0.5 and iqr(p.alpha) < 0.5)))
    return out

S = seed_of("fig4-model-heavy-tails")
rows, seqs = [], {}
t1 = time.time()
for off in OFFSETS:
    for j, s in enumerate(SERIES):
        rng = np.random.default_rng(S + off + j)  # one generator per series: seed, seed + 1, seed + 2
        t = bt.simulate_incremental(N, s, GAMMA, rng)
        if off == 0:
            seqs[s] = t
        for g in (("a", "b") if off == 0 else ("a",)):
            # unbounded exponent (adopted) under every seed; the capped variant (powerlaw 2.0.0 default) under the claim's seed, grid (a)
            rd = bt.ADOPTED_READINGS + (bt.CAPPED_READINGS if (off == 0 and g == "a") else ())
            res = bt.analyse_sequence(t, g, ratios=RATIOS[s], readings=rd)
            for row in judge_series(res):
                rows.append(dict(seed=S + off, series=s, grid=g, lo=round(res["lo"], 2), hi=round(res["hi"], 1), **row))
    print(f"seed offset {off} done, {(time.time() - t1) / 60:.1f} min")
H = pd.DataFrame(rows)
print(H[H.seed == S].to_string())
by = H.groupby(["seed", "grid", "series"]).holds.any().unstack("series")
by["all_series"] = by.all(axis=1)
print("\nholds under at least one reading, per series:"); print(by.to_string())
one = H.groupby(["seed", "grid", "reading"]).holds.all().unstack("reading")
print("\nholds for each series, by reading and seed (grid a):"); print(H[H.grid == "a"].pivot_table(index=["seed", "series"], columns="reading", values="holds", aggfunc="first").to_string())
print("\nexponential series, gamma interquartile range by reading and seed (grid a):"); print(H[(H.grid == "a") & (H.series == "exponential")].pivot_table(index="seed", columns="reading", values="gamma_iqr").to_string())
print("\nholds for all three series under the same reading:"); print(one.to_string())

**Result: partially reproduced under the claim's seed, and seed sensitive.** With the exponent
unbounded, no single reading of the fit holds for all three series under the claim's seed, on
either grid. Under the discrete KS fit the linear and quadratic series hold on grid (a), and the
exponential series has heavy tails (supported at 8 of 11 timescales and 17 of 20 pairs on grid
(a), 42 of 52 and 79 of 94 on grid (b)) but its gamma has an interquartile range of 0.544 and
0.508, above 0.5; on grid (b) the linear series also fails (gamma interquartile range 0.511). The
fits from xmin = 1 hold for the exponential series but not the quadratic one (beta interquartile
range 0.508 to 0.953). Capped at 3 (grid (a)), the discrete KS fit gives 0.522 for the
exponential gamma and still fails. Under the four further seeds the discrete KS fit holds for all
three series under seeds + 1000, + 2000 and + 3000, and not under + 4000, where the linear series
fails; the exponential gamma interquartile range runs from 0.203 to 0.544 over the five seeds.
Taking for each series whichever reading holds, every series holds under every seed on grid (a);
the claim is judged under one reading for all three series. The criterion defines no partial
outcome. Partially reproduced is the protocol's outcome for a result that holds only in part, and
here the criterion holds for two of its three series under the adopted reading. This differs from
fig2-neuron-beta-stable, which is not reproduced because no reading meets its single condition.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(11, 3))
for ax, s in zip(axs, SERIES):
    tau = np.diff(seqs[s]); lo, hi = bt.relevant_range(tau)
    for d in bt.grid(lo, hi, "a")[::2]:
        b = bt.burst_sizes(tau, d); x = np.unique(b)
        ax.loglog(x / b.mean(), bt.ccdf_at(b, x), lw=0.8, label=f"{d:.0f}")
    ax.set_title(s); ax.set_xlabel("b / <b>"); ax.legend(fontsize=6)
axs[0].set_ylabel("P(B >= b)")
print("burst-size CCDFs at every second point of grid (a), claim's seed; all values are plotted")

## fig4-model-iet-shapes

> The IET distribution of the model depends on the timescale series. With linear timescales it
> resembles that of the earthquakes, and with exponential timescales it is a power law
> reminiscent of the Wikipedia edits.

Criterion: the tau_max labels of Figure 4 (1960, 20 800 and 81 600, each +/- 10%), from
simulations set up as in fig4-model-heavy-tails. The comparison of shapes is made by eye in the
paper and is reported, not judged; the earthquake and Wikipedia sequences are not available here
(see fig2-earthquake-beta and fig2-wikipedia-beta), so only the model's shapes are shown. The
largest IET is drawn from the highest level the sequence reaches, so it is shown with that level.

In [ ]:
S2 = seed_of("fig4-model-iet-shapes")
EXP = {"linear": 1960, "quadratic": 20800, "exponential": 81600}
rows = []
iet = {}
for off in OFFSETS:
    for j, s in enumerate(SERIES):
        t = bt.simulate_incremental(N, s, GAMMA, np.random.default_rng(S2 + off + j))
        tau = np.diff(t); d = bt.timescales(s, 400)
        top = int(np.searchsorted(d, tau.max()) - 1)
        rows.append(dict(seed=S2 + off, series=s, tau_min=round(tau.min(), 3), tau_max=round(tau.max(), 1), top_level=top,
                         level_range=f"{d[top]:.0f}-{d[top + 1]:.0f}", expected=EXP[s], rel_diff=round(tau.max() / EXP[s] - 1, 3), within_10pct=bool(abs(tau.max() - EXP[s]) <= 0.1 * EXP[s])))
        if off == 0:
            iet[s] = tau
T = pd.DataFrame(rows); print(T.to_string())
print("\nexpected level of the paper's tau_max:", {s: int(np.searchsorted(bt.timescales(s, 400), v) - 1) for s, v in EXP.items()})
print("within 10% under the claim's seed:", T[T.seed == S2].set_index("series").within_10pct.to_dict())
print("within 10% over the five seeds:", T.groupby("series").within_10pct.sum().to_dict())
fig, ax = plt.subplots(figsize=(4.5, 3))
for s in SERIES:
    h, e = np.histogram(iet[s], np.logspace(0, 6, 61), density=True)
    ax.loglog(np.sqrt(e[1:] * e[:-1]), np.where(h > 0, h, np.nan), "o-", ms=2, label=s)
    print(s, "IETs outside 1 to 1e6:", int(((iet[s] < 1) | (iet[s] > 1e6)).sum()))
ax.set_xlabel("tau"); ax.set_ylabel("P(tau)"); ax.legend(fontsize=7)
# shape: a straight line in log-log over decades marks a power law; slope by least squares on the binned density
for s in SERIES:
    h, e = np.histogram(iet[s], np.logspace(0, 6, 61), density=True); x = np.sqrt(e[1:] * e[:-1]); ok = h > 0
    sl = np.polyfit(np.log10(x[ok & (x > 10)]), np.log10(h[ok & (x > 10)]), 1)[0] if (ok & (x > 10)).sum() > 2 else np.nan
    print(f"{s}: log-log slope of P(tau) above tau = 10: {sl:.2f}; decades with IETs: {np.log10(iet[s].max() / iet[s].min()):.2f}")

**Result: partially reproduced, and seed sensitive.** Under the claim's seed tau_max is 1711.4
(linear, rel_diff -0.127 from 1960), 21 390.9 (quadratic, within 10% of 20 800) and 61 320.3
(exponential, rel_diff -0.249 from 81 600). tau_max is an IET of the highest level the sequence
reaches, which varies from seed to seed (levels 14 to 18 linear, 14 to 16 quadratic, 13 to 18
exponential, against 19, 14 and 16 for the paper's values); over the five seeds the values are
within 10% in 2, 2 and 0 of 5. tau_min is 1.000 or 1.001 in all, as in Figure 4. On shape, P(tau)
falls with a log-log slope of -2.27, -1.98 and -2.15 above tau = 10 over 3.23, 4.33 and 4.79
decades; the comparison with the earthquake and Wikipedia sequences cannot be made here. The criterion
defines no partial outcome; partially reproduced is used, as the protocol defines it, because one
of the three expected values is within its tolerance and two are not.

## model-decremental-equivalent

> The decremental algorithm (Algorithm 1) produces sequences statistically equivalent to those
> of the incremental algorithm (Algorithm 2), since the memory function of Eq. 9 encodes the
> whole merging-number distribution.

Criterion: both algorithms with the parameters of fig4-model-heavy-tails. Reproduced when, at
each relevant timescale, two-sample Kolmogorov-Smirnov tests of the burst sizes do not differ at
p < 0.05 after Bonferroni correction. The relevant timescales are taken from the incremental
sequence, on both grids of O4.

Section IV gives the decremental algorithm P(k), the zeta distribution of Eq. 15, and the
incremental one the memory function of Eq. 16, which it calls the one that Eq. 9 gives. The
sanity cell shows that they differ (q(1) = 0.354 against 0.255), so the two algorithms run
"with the parameters" can be read three ways: (L) literally, zeta for Algorithm 1 and Eq. 16
for Algorithm 2; (Z) both from the zeta distribution, Algorithm 2 with Eq. 9 applied exactly;
(Q) both from Eq. 16, Algorithm 1 with the P(k) whose memory function is Eq. 16. All three are
run. The first generator of the claim drives Algorithm 2, the second (seed + 1) Algorithm 1.
As the test's reference, (N) compares two runs of Algorithm 2 (Eq. 16) with the two generators:
the same algorithm, so any rejection there is the test's own rate on these correlated sequences.

In [ ]:
S3 = seed_of("model-decremental-equivalent")
VARIANTS = {"L": ("eq16", "zeta"), "Z": ("eq9_zeta", "zeta"), "Q": ("eq16", "eq16"), "N": ("eq16", None)}
rows = []
RUNS = {}  # IETs of every run, kept for the calibrated test below
t1 = time.time()
for off in OFFSETS:
    for v, (mem, dist) in VARIANTS.items():
        for j, s in enumerate(SERIES):
            ti = bt.simulate_incremental(N, s, GAMMA, np.random.default_rng(S3 + off + 2 * j), memory=mem)
            rng2 = np.random.default_rng(S3 + off + 2 * j + 1)
            td = bt.simulate_incremental(N, s, GAMMA, rng2) if dist is None else bt.simulate_decremental(N, s, GAMMA, rng2, dist=dist)
            ta, tb = np.diff(ti), np.diff(td)
            RUNS.setdefault((v, s), {"inc": [], "other": []}); RUNS[(v, s)]["inc"].append(ta); RUNS[(v, s)]["other"].append(tb)
            lo, hi = bt.relevant_range(ta)
            for g in ("a", "b"):
                gr = bt.grid(lo, hi, g)
                ps = []; ds = []
                for d in gr:
                    r = ks_2samp(bt.burst_sizes(ta, d), bt.burst_sizes(tb, d))
                    ps.append(r.pvalue); ds.append(r.statistic)
                ps = np.array(ps)
                rows.append(dict(seed=S3 + off, variant=v, series=s, grid=g, n_tests=len(gr), n_reject=int((ps * len(gr) < 0.05).sum()),
                                 min_p_bonf=float(min(1, ps.min() * len(gr))), max_D=round(max(ds), 4),
                                 m_inc=len(bt.burst_sizes(ta, gr[0])), m_dec=len(bt.burst_sizes(tb, gr[0])),
                                 iet_max_inc=round(ta.max(), 0), iet_max_dec=round(tb.max(), 0)))
    print(f"seed offset {off} done, {(time.time() - t1) / 60:.1f} min")
K = pd.DataFrame(rows)
print(K[K.seed == S3].to_string())
eq = K.groupby(["seed", "variant", "grid"]).n_reject.sum().unstack(["variant", "grid"])
print("\nnumber of timescales where the burst sizes differ (Bonferroni p < 0.05), summed over the three series:")
print(eq.to_string())
print(f"\ntime so far {(time.time() - T0) / 60:.1f} min, peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

**Result: not reproduced.** Under the literal set-up (L) and the claim's seed, the burst sizes
differ at every relevant timescale of all three series (23 of 23 on grid (a), 113 of 113 on grid
(b)), and at 22 to 25 and 105 to 114 timescales under the five seeds, with KS D up to 0.4487; the decremental sequences also have more bursts at the lowest
timescale (566 721 against 414 726 in the linear series). With both algorithms on the same P(k)
the differences are small (D at most 0.1874) but some tests still reject under the claim's seed
(Z: 3 and 11 timescales, Q: 4 and 11), as they do for two runs of the same algorithm (N: 4 and
11). The KS criterion thus rejects even identical algorithms on these correlated sequences, and
only Z under seed + 4000 passes it (0 and 0).

### Variant: a test calibrated across independent runs

The criterion's KS test treats every burst of one run as an independent draw, which the
reference N shows is not so: two runs of the same algorithm differ at p as low as 10^-164. As a
variant asked for by the verifier (not the criterion), each algorithm is taken as five
independent runs (the five seeds), and runs, not bursts, are the units of the test. At each
point of grid (a) of the claim's-seed incremental run, the KS distance D is taken between every
two of the ten runs. The statistic is the mean D between runs of different algorithms less the
mean D between runs of the same algorithm, averaged over the timescales. Its null is the 126
distinct splits of the ten runs into two groups of five (an exact permutation test; the
smallest possible p is 1/126 = 0.0079). Bonferroni over the three series. N is the check that
the test does not reject two runs of one algorithm.

In [ ]:
from itertools import combinations
splits = [set(c_) for c_ in combinations(range(10), 5) if 0 in c_]  # 126 distinct splits
perm = []
for (v, s), rr in RUNS.items():
    runs = rr["inc"] + rr["other"]
    lo, hi = bt.relevant_range(runs[0])
    G = [d for d in bt.grid(lo, hi, "a") if all(len(bt.burst_sizes(x, d)) >= bt.MIN_BURSTS for x in runs)]
    Dm = np.zeros((len(G), 10, 10))
    for gi, d in enumerate(G):
        cd = [bt.cdf_int(bt.burst_sizes(x, d)) for x in runs]
        for i, j in combinations(range(10), 2):
            Dm[gi, i, j] = Dm[gi, j, i] = bt.ks_d_int(cd[i], cd[j])
    Dbar = Dm.mean(axis=0)
    def stat(A):
        B_ = [i for i in range(10) if i not in A]; A = sorted(A)
        between = Dbar[np.ix_(A, B_)].mean()
        within = np.mean([Dbar[i, j] for grp in (A, B_) for i, j in combinations(grp, 2)])
        return between - within
    obs = stat({0, 1, 2, 3, 4}); null = np.array([stat(A) for A in splits])
    p = float((null >= obs - 1e-15).mean())
    perm.append(dict(variant=v, series=s, n_timescales=len(G), D_between=round(Dbar[:5, 5:].mean(), 4),
                     D_within_inc=round(Dbar[:5, :5][np.triu_indices(5, 1)].mean(), 4), D_within_other=round(Dbar[5:, 5:][np.triu_indices(5, 1)].mean(), 4),
                     stat=round(obs, 4), p=round(p, 4), p_bonf=round(min(1, 3 * p), 4), differ=bool(3 * p < 0.05)))
PM = pd.DataFrame(perm); print(PM.to_string())
print("\nseries where the runs differ (Bonferroni p < 0.05), by variant:", PM.groupby("variant").differ.sum().to_dict())
del RUNS
print(f"\nnotebook time {(time.time() - T0) / 60:.1f} min, peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

**Result (calibrated variant).** With the five runs of each algorithm as the units, the literal
set-up (L) differs in all three series (statistic 0.169 to 0.236, p = 0.0079, the smallest
possible, 0.024 after Bonferroni), while two runs of one algorithm (N) differ in none (p 0.373 to
0.913), so this test keeps its level where the criterion's does not. With both algorithms on the
zeta distribution (Z) no series differs (p 0.429 to 0.857). With both on Eq. 16 (Q) the linear
series differs (p = 0.0079; mean D 0.0078 between the algorithms against 0.0077 and 0.0053 within
each) and the other two do not. The variant agrees with the criterion on L, and it finds the two
algorithms equivalent under Z, as the paper says they are when both use one distribution; it does
not decide the outcome, which the criterion gives.